# PumpWatch exploratory analysis
Real Binance data and independently published onset labels. Training-only exploration; held-out results are reported separately. Run from this project folder.

In [ ]:
# Load saved historical predictions and inspect the chronological partitions.
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
root = Path.cwd()
if not (root / 'artifacts').exists():
    root = root.parent
d = pd.read_csv(root / 'artifacts/predictions.csv.gz')
report = json.loads((root / 'artifacts/report.json').read_text())
pd.DataFrame(report['splits']).T

## Class balance and feature distributions
Synthetic test fixtures never enter this analysis. Normal means no known event. Excluded neighborhoods have target -1.

In [ ]:
# Count windows by partition and class; -1 denotes excluded ambiguous windows.
d.groupby(['split','target']).size().rename('windows').to_frame()

In [ ]:
# Compare feature medians using training data only, avoiding test-informed exploration.
training = d[(d.split == 'train') & (d.target >= 0)]
training.groupby('target')[['return_1m','volume_z','volatility_15m','trade_z']].median()

In [ ]:
# Visualize volume anomalies in a reproducible sample of training windows.
sample = pd.concat([training.query('target == 1'), training.query('target == 0').sample(3000, random_state=42)])
px.box(sample, x='target', y='volume_z', color='target', points=False, title='Training volume anomalies')

## Held-out comparison
The model and thresholds were selected before final test evaluation.

In [ ]:
# Display held-out test metrics for the four models without fitting again.
pd.read_csv(root / 'artifacts/model_comparison.csv')

In [ ]:
# Inspect independent-event detections and close-time delays in the test period.
pd.read_csv(root / 'artifacts/event_results.csv')

## Interpretation
Scores are uncalibrated. Event-conditioned sampling, historical regimes and uncertain labels limit generalization. See Research_Report.html and artifacts/errors.csv for the error analysis.